# ZiguratIP on Google Colab

Build and run **ZiguratIP** — a single-process database (*Zigurat*), language (*Parsi*),
and web server (*Zeytun*) written in dependency-free C++11 — inside a Colab VM, and reach
its HTTP front end from your browser.

- **Zigurat** — MVCC storage engine, binary protocol on port **2160**
- **Parsi** — SQL-like language compiled to a `.so` the server `dlopen`s
- **Zeytun** — HTTP server for static files and `.zt` pages on port **2190**

> ⚠️ **Run this as a throwaway sandbox only.** ZiguratIP hand-rolls its own crypto. The
> branch used here fixes the two worst problems — RSA keys were seeded from `time(0)`, so two
> `ca keygen` runs in the same second produced *byte-identical* private keys, and the Parsi
> compiler was reachable over the network, which is arbitrary code execution as the server's
> user. Both are fixed, but plenty remains: remotely-triggerable out-of-bounds reads in the
> HTTP parser, a stack-smashing VLA on large static files, unbounded allocations, XSS by
> construction, and no write-ahead log. Never point it at real data, never leave the proxied
> URL up, and keep TLS **off** (a browser can't negotiate its static-RSA suites anyway —
> Colab's proxy gives you HTTPS on the outside).
>
> Any key material generated *before* this branch is compromised and must be reissued.

**How it maps to Colab:** Colab is the *compute*. Google Drive is only *storage* — nothing
runs "on" Drive. We therefore **build and run on the VM's local disk** and use Drive purely
for an optional copy-in / copy-out snapshot of the data directory. Drive's FUSE mount is
hostile to the storage engine's random in-place writes and to `dlopen`, so the live database
must never sit on the mount.

## 1 · Build

> **Use the branch below, not `master`.** ZiguratIP did not build on Linux at all: fixed-width
> integer types missing `<cstdint>`, `htonl` and friends coming back as glibc macros, and two
> libraries that never declared what they link against. macOS hid all three. Worse, every recipe
> in the top-level `Makefile` is prefixed with `@-`, so failures were stepped over and the run
> still ended with `******* all done *******` — a clean checkout produced 2 of 14 libraries and
> no executables while reporting success. The branch fixes that; `master` will still appear to
> build and then have nothing to run.

Colab already ships `g++` and `make`; the `apt-get` line is just insurance.

In [ ]:
import os, subprocess

REPO   = 'https://github.com/saman-pasha/ZiguratIP.git'
BRANCH = 'colab'   # all fixes live here; master does not build on Linux
SRC    = '/content/ZiguratIP'

!apt-get -qq install -y build-essential >/dev/null 2>&1 || true

# Check the branch exists before cloning. A clone of a branch that is not
# there fails quietly enough that every later cell blames something else.
ls = subprocess.run(['git','ls-remote','--heads',REPO,BRANCH],
                    capture_output=True, text=True)
assert BRANCH in ls.stdout, f'branch {BRANCH!r} not found on the remote:\n{ls.stdout}{ls.stderr}'
print(f'branch {BRANCH} found')

# Clone, or bring an existing clone up to date. Re-running this notebook
# in a session that already cloned used to skip straight past here and
# rebuild whatever was fetched the first time, so a fix pushed since then
# never arrived and the symptom it fixed was still there.
if not os.path.isdir(SRC):
    !git clone --depth 1 -b $BRANCH $REPO $SRC
else:
    !git -C $SRC fetch --depth 1 origin $BRANCH
    !git -C $SRC checkout -B $BRANCH FETCH_HEAD
assert os.path.isfile(f'{SRC}/Makefile'), 'clone produced no tree'

head = subprocess.run(['git','-C',SRC,'log','-1','--format=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print('building:', head)

# A rebuild has to see the new sources, and the object files from the
# previous run are older than nothing the Makefile knows to check.
!make -C $SRC clean >/dev/null 2>&1 || true

%cd /content/ZiguratIP
!make MODE=Release 2>&1 | tail -n 5

# make exits 0 even when projects fail -- every recipe in the top-level
# Makefile is prefixed with @-, so failures are stepped over and it still
# prints 'all done'. Check what was actually produced instead.
libs = sorted(f for f in os.listdir('home/lib') if f.endswith('.so'))
bins = sorted(os.listdir('home/bin'))
print(f'\nlibraries: {len(libs)} (expect 14)')
print(f'executables: {bins}')
missing = {'Test','ca','parsi','ziguratip'} - set(bins)
assert not missing and len(libs) >= 14, f'BUILD INCOMPLETE -- missing {missing or "libraries"}'
print('build OK')

## 2 · Runtime environment

`ZIGURATIP_HOME` is both the install prefix and the runtime home. On **Linux** the shared
libraries are found via `LD_LIBRARY_PATH` (not macOS's `DYLD_LIBRARY_PATH`). A C++ compiler
must also stay on `PATH` — Parsi pages are compiled to a `.so` *at request time*, not only
at build time.

In [ ]:
os.environ['ZIGURATIP_HOME']  = '/content/ZiguratIP/home'
os.environ['LD_LIBRARY_PATH'] = os.environ['ZIGURATIP_HOME'] + '/lib'
assert subprocess.call(['which', 'c++']) == 0, 'a C++ compiler must be on PATH for runtime Parsi compilation'
print('ZIGURATIP_HOME =', os.environ['ZIGURATIP_HOME'])
print('binaries:')
!ls -1 $ZIGURATIP_HOME/bin

## 3 · (Optional) restore data from Google Drive

Colab VMs are ephemeral — `home/data` is lost when the runtime recycles. To keep a database
between sessions, set `PERSIST = True`. We copy a saved snapshot from Drive **onto local disk
before starting** the server; we never run the engine directly on the Drive mount.

Leave `PERSIST = False` for a clean, throwaway run (the store is created on first use).

In [ ]:
PERSIST = False  # set True to keep the database across sessions via Drive
DRIVE_BACKUP = '/content/drive/MyDrive/ziguratip-backup/data'

if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.isdir(DRIVE_BACKUP):
        print('restoring data snapshot from Drive ...')
        !rm -rf $ZIGURATIP_HOME/data
        !cp -a "$DRIVE_BACKUP" $ZIGURATIP_HOME/data
        print('restored.')
    else:
        print('no Drive snapshot yet — a fresh store will be created on first use.')
else:
    print('PERSIST is off — using a throwaway store on local disk.')

## 4 · Build the demo objects

`demo/build.sh` compiles the demo tables and `.zt` pages with the **offline** `parsi` compiler.
It starts nothing — it just produces the shared objects the server will load.

Note this is the supported way to compile. Compiling *over the network* is refused by default
(`COMPILER/REMOTE_MODE`), because it runs a C++ compiler and linker on whatever a client sends,
which is arbitrary code execution as the server's user. Leave it off.

In [ ]:
!chmod +x demo/build.sh Test/*.sh
!./demo/build.sh 2>&1 | tail -n 12

## 5 · Start the server (background)

The server blocks and listens forever, so it can't own a notebook cell — we launch it as a
background process and tail its log. It prints what it loaded, then listens on **2160**
(binary) and **2190** (HTTP).

In [ ]:
import time

# stop a previous instance if this cell is re-run
try:
    srv.terminate(); srv.wait(timeout=5)
except Exception:
    pass

srv = subprocess.Popen(['./home/bin/ziguratip'],
                       stdout=open('server.log', 'w'),
                       stderr=subprocess.STDOUT,
                       env=os.environ)
time.sleep(2.5)
print(open('server.log').read())
assert srv.poll() is None, 'server exited — see the log above'

## 6 · Reach the HTTP server from your browser

Colab does not expose raw ports to the internet, but it has a built-in port proxy. The cell
below returns a clickable HTTPS URL that front-ends port **2190**.

The binary protocol on 2160 is *not* browser-reachable — you would only use it from a
Connector client running inside this same VM.

Demo pages to try (append to the proxy URL): `/setup.zt` creates rows, `/catalog.zt` browses
them, `/lookup.zt`, `/bulk.zt`, `/report.zt`.

In [ ]:
import urllib.request, time

# Prove the server answers HERE before handing out a proxy URL.
# proxyPort() returns a URL whether or not anything is listening, so a dead
# server and a healthy one look identical from the browser: both render a
# blank page. Check locally first and say so plainly if it is not up.
code = None
for attempt in range(10):
    try:
        with urllib.request.urlopen('http://127.0.0.1:2190/', timeout=5) as r:
            code, body = r.status, r.read()
        break
    except Exception as e:
        err = e; time.sleep(1)

if code is None:
    print('SERVER IS NOT ANSWERING on 127.0.0.1:2190 --', err)
    print('--- server.log ---'); print(open('server.log').read()[-2000:])
    raise SystemExit('not starting the proxy: there is nothing behind it')

print(f'local check: HTTP {code}, {len(body)} bytes')
assert code == 200 and len(body) > 0, 'server answered but served nothing'

# Render it INSIDE the notebook first. This needs no proxy at all, so it
# works even when the proxy URL does not, and it is the fastest way to see
# that the server is really serving.
from IPython.display import HTML, display
display(HTML(body.decode('utf-8', 'replace')))

# Then the browser-reachable view. serve_kernel_port_as_iframe embeds the
# port in this output cell through Colab's authenticated routing.
# (serve_kernel_port_as_window is deliberately not used: Colab itself warns
# it may stop working as browsers tighten cross-origin rules.) Prefer it to a bare
# proxyPort URL, which is only valid inside the session that minted it: a
# URL kept from an earlier runtime answers 404 from Colab's edge, before
# any of this is reached, and the server never sees the request.
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(2190, height=640)

from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(2190)')
print('Direct URL (this session only):', url)
print('Demo:', url.rstrip('/') + '/setup.zt', '(run once, then /catalog.zt)')

## 7 · (Optional) snapshot data back to Drive

Only meaningful when `PERSIST = True` **and** after a clean stop (next cell). ZiguratIP has no
write-ahead log, so a snapshot taken while the server is mid-write can be inconsistent — stop
the server first, then run this.

In [ ]:
if PERSIST:
    !mkdir -p "$(dirname "$DRIVE_BACKUP")"
    !rm -rf "$DRIVE_BACKUP"
    !cp -a $ZIGURATIP_HOME/data "$DRIVE_BACKUP"
    print('snapshot written to', DRIVE_BACKUP)
else:
    print('PERSIST is off — nothing to save.')

## 8 · Stop the server

In [ ]:
try:
    srv.terminate(); srv.wait(timeout=5)
    print('server stopped.')
except Exception as e:
    print('nothing running:', e)

## Notes & limits

- **Durability:** the storage engine has no WAL/fsync, so a hard runtime kill mid-commit can
  corrupt the store. Snapshot to Drive only after a clean stop.
- **Concurrency:** thread-per-connection on a pool of 5 — a handful of keep-alive browser tabs
  can saturate it. Fine for a demo, not for load.
- **TLS:** leave `HTTP/TLS_MODE: FALSE` (the default in `home/etc/ziguratip.conf`). The server
  only offers static-RSA suites no browser will accept; Colab's proxy already gives you HTTPS.
- **Drive:** never set `HOME_PATH`/data onto the `/content/drive` mount — FUSE breaks the
  pager's random in-place writes and `dlopen`. Local disk for running, Drive for snapshots.
- **Security:** guessable RSA keys and a network-reachable compiler make this unsafe to expose.
  Treat every run as disposable.